# Moebius — Sphere-Congruence Quad Mesh Pipeline

Full pipeline in six stages:
0. **Setup** — imports, env check
1. **Load surface** — B-spline JSON, u/v range, flip options
2. **Initialisation** — sphere radii r(u,v), line congruence l(u,v)
3. **LC optimisation** — minimise l·cu, l·cv; enforce angle with n
4. **Torsal optimisation** — add torsal-plane constraints
5. **Remeshing** — export cross-field, run remesher, load result
6. **Post-optimisation** — sphere fit, support planarity, face regularity
7. **Save results** — OBJ files, sphere panels, state pickle

In [37]:
import sys, os
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

CHAKANA = (ROOT / "Chakana_Geo").resolve()
for sub in ["hanan/src", "kayviz/src"]:
    p = str(CHAKANA / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import matplotlib.pyplot as plt
import kayviz as kv

from moebius.pipeline import (
    MoebiusState, load_surface, sample_selection, compute_init,
    setup_bspline_optimizer, run_bspline_optimizer,
    setup_lc_optimizer, run_lc_optimizer,
    setup_torsal_optimizer, run_torsal_optimizer, recompute_torsal_directions,
    check_remesher, export_frame_field, torsal_field_per_quad,
    run_remesher, load_remeshed_surface, remesh_quality, sweep_gradient,
    fair_remeshed_surface,
    setup_postopt_optimizer, run_postopt_optimizer,
    save_results, load_state,
)
from hanan.geometry.io import read_obj, write_obj
from moebius.utils.bsplines import sample_bspline_surface, bspline_curvatures
from moebius.glyphs import panels_from_spheres
from moebius.utils.spheres import center_radius_from_implicit
from scipy.interpolate import bisplev

kv.init()
kv.clear()

---
## 1 — Load B-spline surface

In [40]:
# ── parameters (edit these) ───────────────────────────────────────────────────
SURFACE_NAME    = "surface_00002"          # JSON filename without extension3
EXPERIMENT_NAME = "experiment0"

# u/v sampling
U_SAMPLE = 60
V_SAMPLE = 20
INTERVAL_U = [0.1, 0.9]
INTERVAL_V = [0.1, 0.9]

# Orientation options
FLIP_NORMALS = True   # reverse u direction
SWAP_UV      = False   # swap u <-> v
# ─────────────────────────────────────────────────────────────────────────────

state = load_surface(
    SURFACE_NAME, EXPERIMENT_NAME,
    u_sample_num=U_SAMPLE, v_sample_num=V_SAMPLE,
    interval_u=INTERVAL_U, interval_v=INTERVAL_V,
)

# Apply flip/swap if requested
if FLIP_NORMALS:
    cp = state.bspline.controlpoints
    state.bspline.controlpoints = cp[::-1, :, :]
    state.normalsFlipped = True
if SWAP_UV:
    state.bspline = state.bspline.clone()
    state.bspline.controlpoints = state.bspline.controlpoints.swapaxes(0, 1)
    state.uvSwapped = True

print(f"Surface: {SURFACE_NAME}")
print(f"Sampling: {U_SAMPLE}×{V_SAMPLE} on u∈{INTERVAL_U}, v∈{INTERVAL_V}")

import ipywidgets as widgets
from IPython.display import display

def _update_surface(u_min, u_max, v_min, v_max, u_n, v_n):
    if u_min >= u_max or v_min >= v_max:
        return
    sample_selection(state,
                     interval_u=[u_min, u_max],
                     interval_v=[v_min, v_max],
                     u_sample_num=u_n,
                     v_sample_num=v_n)
    V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
    K, H, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
    kv.register_surface_mesh("surface", V, F)
    kv.add_scalar_quantity("surface", "MeanCurvature",     H.ravel(), defined_on="vertices")
    kv.add_scalar_quantity("surface", "GaussianCurvature", K.ravel(), defined_on="vertices")

_sl = dict(continuous_update=False, style={"description_width": "80px"}, layout=widgets.Layout(width="340px"))
u_min_w = widgets.FloatSlider(value=0.0, min=0.0,  max=0.95, step=0.05, description="u min", **_sl)
u_max_w = widgets.FloatSlider(value=1.0, min=0.05, max=1.0,  step=0.05, description="u max", **_sl)
v_min_w = widgets.FloatSlider(value=0.0, min=0.0,  max=0.95, step=0.05, description="v min", **_sl)
v_max_w = widgets.FloatSlider(value=1.0, min=0.05, max=1.0,  step=0.05, description="v max", **_sl)
u_n_w   = widgets.IntSlider(value=U_SAMPLE, min=5, max=60, step=5, description="u samples", **_sl)
v_n_w   = widgets.IntSlider(value=V_SAMPLE, min=5, max=60, step=5, description="v samples", **_sl)

out = widgets.interactive_output(_update_surface, {
    "u_min": u_min_w, "u_max": u_max_w,
    "v_min": v_min_w, "v_max": v_max_w,
    "u_n":   u_n_w,   "v_n":   v_n_w,
})
ui = widgets.VBox([
    widgets.HBox([u_min_w, u_max_w]),
    widgets.HBox([v_min_w, v_max_w]),
    widgets.HBox([u_n_w,   v_n_w]),
])
display(ui, out)
kv.show()

Surface: surface_00002
Sampling: 60×20 on u∈[0.1, 0.9], v∈[0.1, 0.9]


Output()

[kayviz] Viewer already open — http://127.0.0.1:8001/?app=script


---
## 1b — B-spline optimisation (enforce H ≠ 0)

Slightly deforms the surface control points to ensure the mean curvature H has
a consistent nonzero sign throughout the domain. This makes the sphere-congruence
initialisation r = 1/H well-conditioned.

Skip this cell if the surface already has |H| > 0 everywhere.

In [ ]:
from moebius.pipeline import setup_bspline_optimizer, run_bspline_optimizer

# ── parameters ────────────────────────────────────────────────────────────────
H_THRESH     = 0.05        # floor: |H| must stay above this
H_MAX        = None       # ceiling on |H|; None = off.  Set it to stop the
                          # floor being answered with a crease - a local dent
                          # is the cheapest way to raise |H| at a flat point.
                          # Absolute curvature, so read the initial max |H|
                          # off the surface and allow ~1x it.
W_MAX        = 1.0        # weight of the ceiling relative to the floor
W_H          = 1.0
W_STEP       = 0.1
BSPLINE_ITER = 30
# ─────────────────────────────────────────────────────────────────────────────

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H_before, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
print(f"H before  min={H_before.min():.4f}  max={H_before.max():.4f}  "
      f"zero-crossings≈{int((np.diff(np.sign(H_before.ravel())) != 0).sum())}")

setup_bspline_optimizer(state, H_thresh=H_THRESH, H_max=H_MAX, w_max=W_MAX,
                        w_H=W_H, w_step=W_STEP)
run_bspline_optimizer(state, max_iter=BSPLINE_ITER)

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H_after, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
print(f"H after   min={H_after.min():.4f}  max={H_after.max():.4f}  "
      f"zero-crossings≈{int((np.diff(np.sign(H_after.ravel())) != 0).sum())}")

kv.register_surface_mesh("surface", V, F)
kv.add_scalar_quantity("surface", "H_before", H_before.ravel(), defined_on="vertices")
kv.add_scalar_quantity("surface", "H_after",  H_after.ravel(),  defined_on="vertices")
kv.show()

---
## 2 — Sphere-congruence initialisation

In [42]:
state.init_mode = 0

compute_init(state)

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
r_vals = bisplev(state.u_pts, state.v_pts, state.r_uv)
n_flat = state.n_surf.reshape(-1, 3)
Vc = V + r_vals.ravel()[:, None] * n_flat

kv.register_surface_mesh("sphere_centers_init", Vc, F)
kv.register_surface_mesh("surface", V, F)
kv.add_vector_quantity("surface", "l_init", state.l_init.reshape(-1, 3), defined_on="vertices")
kv.show()

Init  r range  [0.1473, 20.0000]
Init  l·n mean  0.2482
[kayviz] Viewer already open — http://127.0.0.1:8001/?app=script


---
## 3a — Line-congruence optimisation

In [43]:
# ── parameters ────────────────────────────────────────────────────────────────
LC_MAX_ITER  = 100
ANGLE_LC_DEG = 20.0   # theta: MAX angle l <-> n. Smaller = stronger.
                      # Paper Tables 1-2 use 15 deg (25 deg for the denser grid).
W_LC         = 2.0
W_ORTH       = 2.0    # paper Table 1: w_LCOrth = 2 in this stage
HINGE        = True   # one-sided residuals, no mu/theta slack variables
# ─────────────────────────────────────────────────────────────────────────────

state.angle_lc = ANGLE_LC_DEG

setup_lc_optimizer(state, w_lc=W_LC, w_orth=W_ORTH, hinge=HINGE)
run_lc_optimizer(state, max_iter=LC_MAX_ITER)
l_opt, cp_opt = state.lc_opt.unpack("l", "rij")
state.r_uv[2] = cp_opt
r_vals = bisplev(state.u_pts, state.v_pts, state.r_uv)
Vc = V + r_vals.ravel()[:, None] * n_flat

l_opt   = l_opt.reshape(-1, 3)
angle_n = np.degrees(np.arccos(np.clip(np.einsum("ij,ij->i", np.abs(l_opt), np.abs(n_flat)), -1, 1)))

kv.register_surface_mesh("surface",        V, F)
kv.register_surface_mesh("sphere_centers", Vc, F)
kv.add_vector_quantity("surface", "l_opt",    l_opt,   defined_on="vertices")
kv.add_scalar_quantity("surface", "angle_l_n", angle_n, defined_on="vertices")
kv.show()

Mesh |V|=1200, |F|=1121, |E|=2320
Adding quadratic fairness for 1044 vertices with 4 neighbors
Adding Laplacian fairness for 156 
Thread pool: 5 workers (terms=5, cpus=32)
Starting fixed-damping LM optimization (max_iter=100, tol=1e-08)
Levenberg-Marquardt Optimization (Simple)
Iter   | Energy        | Norm Energy   | ||dx||        | μ             | ρ        | Time    
------------------------------------------------------------------------------------------
1      | 1.546393e+03  | 1.256966e+00  | 2.601751e+02  | 2.270437e-04  | N/A      | 1.2142  
10     | 3.004302e+03  | 1.914838e-02  | 7.466660e+00  | 2.270437e-04  | N/A      | 4.6747  
20     | 4.426493e+00  | 6.256937e-04  | 1.689155e+00  | 2.270437e-04  | N/A      | 0.7111  
30     | 4.611495e-01  | 2.225950e-04  | 8.168686e-01  | 2.270437e-04  | N/A      | 0.6774  
40     | 1.073109e-07  | 8.029127e-11  | 4.282986e-03  | 2.270437e-04  | N/A      | 0.7210  
Converged at iteration 42 with energy 7.812534e-09

 WEIGHTS REPORT
Term

---
## 3b — Torsal optimisation

In [44]:
# ── parameters ────────────────────────────────────────────────────────────────
TORSAL_MAX_ITER  = 100
ANGLE_TORSAL_DEG = 75.0   # alpha: MINIMUM angle between the torsal PLANES.
                          # Larger = stronger (90 = exactly orthogonal planes);
                          # 45 is the minimum useful value. Paper used 45 / 60.
W_TORSAL         = 1.0
W_TANGLE         = 2.0    # paper Table 2: w_angle = 2
W_ORTH_TORSAL    = 0.1   # paper Table 2: w_LCOrth = 0.01 here -- two orders
                          # below stage 2 on purpose. Holding the angle
                          # constraint hard at this stage drags l off
                          # unit(cu x cv) instead of letting the radii move.
RECOMPUTE_STEPS  = 10     # paper Table 2 "RS": re-solve the analytic torsal
                          # directions every N steps, so the optimisation does
                          # not stall on degenerate directions.
FINAL_PLAIN      = 10     # closing iterations with w_angle = 0
# ─────────────────────────────────────────────────────────────────────────────

state.angle_torsal = ANGLE_TORSAL_DEG

setup_torsal_optimizer(state, w_lc=W_LC, w_orth=W_ORTH_TORSAL,
                        w_torsal=W_TORSAL, w_tangle=W_TANGLE)
run_torsal_optimizer(state, max_iter=TORSAL_MAX_ITER,
                      recompute_steps=RECOMPUTE_STEPS,
                      final_plain_steps=FINAL_PLAIN)

Mesh |V|=1200, |F|=1121, |E|=2320
Adding quadratic fairness for 1044 vertices with 4 neighbors
Adding Laplacian fairness for 156 
Thread pool: 11 workers (terms=11, cpus=32)
Starting fixed-damping LM optimization (max_iter=10, tol=1e-08)
Levenberg-Marquardt Optimization (Simple)
Iter   | Energy        | Norm Energy   | ||dx||        | μ             | ρ        | Time    
------------------------------------------------------------------------------------------
1      | 1.565792e+02  | 1.249536e-01  | 2.289744e+00  | 3.539079e+00  | N/A      | 3.2353  
10     | 3.395518e+01  | 1.126463e-02  | 1.998795e-01  | 3.539079e+00  | N/A      | 3.3700  
Reached maximum iterations (10)
Starting fixed-damping LM optimization (max_iter=20, tol=1e-08)
20     | 1.575612e+01  | 5.835235e-03  | 9.112407e-02  | 3.539079e+00  | N/A      | 2.5783  
Reached maximum iterations (20)
Starting fixed-damping LM optimization (max_iter=30, tol=1e-08)
30     | 1.151573e+01  | 4.514441e-03  | 7.259564e-02  | 3.539079

In [45]:
from moebius.utils.mesh_utils import vec_dot
from moebius.pipeline import torsal_field_per_quad

l_t, nt1, nt2 = state.torsal_opt.unpack("l", "nt1", "nt2")
l_t = l_t.reshape(state.u_sample_num, state.v_sample_num, 3)

# t_k = u_k·du + v_k·dv in the SURFACE tangent basis of each quad
# (du = v2-v0, dv = v1-v3), the same convention as energies.Torsal.
t1_n, t2_n = torsal_field_per_quad(state)

nt1 = nt1.reshape(-1, 3)
nt2 = nt2.reshape(-1, 3)
torsal_angle = np.degrees(np.arccos(np.clip(np.abs(vec_dot(nt1, nt2)), 0, 1)))

# Angle between the two torsal directions — near 0 means a degenerate frame,
# which is what makes the remesher produce singularities.
t_angle = np.degrees(np.arccos(np.clip(np.abs(vec_dot(t1_n, t2_n)), 0, 1)))
print(f"angle(t1,t2)   min {t_angle.min():.1f}°  mean {t_angle.mean():.1f}°  max {t_angle.max():.1f}°")
print(f"degenerate (<5°): {int((t_angle < 5).sum())} / {len(t_angle)} quads")

# Face barycenters, arrows scaled to ~1/4 of the mean face diagonal
bary = (V[F[:, 0]] + V[F[:, 1]] + V[F[:, 2]] + V[F[:, 3]]) / 4
diag = np.mean(np.linalg.norm(V[F[:, 2]] - V[F[:, 0]], axis=1))
s = diag / 4

# Each face contributes one centred segment  [bary - s*t, bary + s*t]
def _torsal_curves(bary, t, scale):
    F_cnt = len(bary)
    verts = np.empty((2 * F_cnt, 3))
    verts[0::2] = bary - scale * t
    verts[1::2] = bary + scale * t
    edges = np.stack([np.arange(0, 2*F_cnt, 2),
                      np.arange(1, 2*F_cnt, 2)], axis=1)
    return verts, edges

v1_net, e1_net = _torsal_curves(bary, t1_n, s)
v2_net, e2_net = _torsal_curves(bary, t2_n, s)

kv.register_surface_mesh("surface", V, F)
kv.add_scalar_quantity("surface", "torsal_angle", torsal_angle, defined_on="faces")
kv.add_scalar_quantity("surface", "t1_t2_angle",  t_angle,      defined_on="faces")
kv.add_vector_quantity("surface", "l", l_t.reshape(-1, 3), defined_on="vertices")

kv.register_curve_network("torsal_t1", v1_net, e1_net, color=(0.9, 0.2, 0.1), radius=0.002)
kv.register_curve_network("torsal_t2", v2_net, e2_net, color=(0.1, 0.3, 0.9), radius=0.002)
kv.show()

angle(t1,t2)   min 0.0°  mean 77.2°  max 90.0°
degenerate (<5°): 2 / 1121 quads
[kayviz] Viewer already open — http://127.0.0.1:8001/?app=script


---
## 4 — Remeshing

`quadRemesher` takes a **triangle** mesh plus a **frame field** (two directions
per face) and integrates the field into a quad mesh whose edges follow it.
`export_frame_field` writes what it needs into `state.save_path`:

| file | contents |
|---|---|
| `remesh_input.obj` | the sampling grid, **triangulated** (quads are rejected) |
| `D1.dat`, `D2.dat` | one unit torsal direction per constrained triangle, `--format vec` |
| `indices.dat` | which triangles those rows belong to (`--indices`) |

A torsal direction lies in the plane of its quad's two **diagonals**. Where the
quad is non-planar neither of its two triangles lies in that plane, so the
remesher's in-plane projection rotates the direction. `export_frame_field`
therefore constrains only **one triangle per quad** — whichever needs the
smaller rotation — and lets the remesher interpolate the rest
(`one_frame_per_quad=True`, the default). On `CMC` at 15² sampling that took the
rotation from 5.2°/49° (mean/max) down to 3.7°/28° and the singularity count
from 11 to 9. Pass `one_frame_per_quad=False` to constrain both triangles.

**Quad density is `--gradient H`** (default 40): *higher = denser*, relative to
the bounding box. `--constraints F` keeps only a fraction `F` of the faces as
hard constraints; lowering it to ~0.1 lets the solver smooth the field and
usually gives fewer singularities.

**Dropping bad frames.** Anything not listed in `indices.dat` is *interpolated*
by the remesher, so a frame that is wrong is better left out than fed in — one
rogue direction locally deflects the quads around it.
`export_frame_field(..., filter_outliers=True)` drops, by default:

- quads whose torsal quadratic had a negative discriminant, i.e. no **real**
  torsal directions exist there and `torsal_directions()` returned a
  least-squares fit (`drop_no_real_torsal`);
- frames disagreeing with their 4 grid neighbours by more than
  `max_neighbour_deg` (35°), measured blind to sign and to t1/t2 ordering,
  scored by the *median* over neighbours so the outlier goes and its
  neighbours stay;
- near-degenerate frames, `angle(t1,t2) < min_frame_angle_deg` (5°).

`boundary_rings=N` additionally drops N rings along the grid border, where `l`
is least constrained and the discriminant most often goes negative. On `Tunel`
every flagged quad was on the outer ring — but the targeted criteria caught
them while dropping ~1% of quads, against 20% for `boundary_rings=1`.

The remesher runs two ways, both via `run_remesher(..., batch=...)`:

- `batch=True` — headless: integrate, extract, write `<name>_Remeshed.obj`, exit.
- `batch=False` — open its viewer: `+`/`-` change the gradient live, `R`
  previews the quads, `S` saves. Use this to find a good `H` by eye, then set
  `GRADIENT` below to that value and re-run in batch so the state holds it.

In [46]:
# ── export the frame field ────────────────────────────────────────────────────
# Found automatically at ../QuadRemesher/bin/quadRemesher or
# ../QuadRemesher/Quad_Remesher/build/quadRemesher; override if it lives elsewhere:
# state.remesher_bin = "/path/to/quadRemesher"
check_remesher(state.remesher_bin or None)

ONE_FRAME_PER_QUAD = True   # constrain 1 of each quad's 2 triangles (see above)
FILTER_OUTLIERS    = True   # leave bad frames out; the remesher interpolates them
BOUNDARY_RINGS     = 0      # also drop N rings along the border (try 1 if needed)
MAX_NEIGHBOUR_DEG  = 35.0   # flag frames this far from their neighbours
MIN_FRAME_ANGLE    = 5.0    # flag near-degenerate frames

field = export_frame_field(state,
                            one_frame_per_quad=ONE_FRAME_PER_QUAD,
                            filter_outliers=FILTER_OUTLIERS,
                            boundary_rings=BOUNDARY_RINGS,
                            max_neighbour_deg=MAX_NEIGHBOUR_DEG,
                            min_frame_angle_deg=MIN_FRAME_ANGLE)
field

Remesher OK: /home/anthony/KAUSTLocal/QuadRemesher/Quad_Remesher/build/quadRemesher
Exported 1200 vertices, 2242 triangles -> /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0
  outlier filter: kept 985/1121 quads (87.9%) as constraints; the remesher interpolates the rest
     dropped     5  no real torsal directions (discriminant < 0)
     dropped     2  degenerate frame
     dropped   136  not smooth (>35 deg from neighbours)
  985 frames on 1 of the 2 triangles of each of 1121 quads
  |angle(t1,t2)|  min 0.0 deg  mean 77.2 deg  max 90.0 deg
  in-plane rotation on the constrained triangles: mean 1.5 deg, max 16.6 deg


{'obj': '/home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0/remesh_input.obj',
 'd1': '/home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0/D1.dat',
 'd2': '/home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0/D2.dat',
 'indices': '/home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0/indices.dat',
 'n_vertices': 1200,
 'n_triangles': 2242,
 'n_frames': 985,
 'n_quads': 1121,
 'angle_t1_t2_deg': {'min': 0.00023478620039850049,
  'mean': 77.20946313617495,
  'max': 89.99962353845814},
 'rotation_deg': {'mean': 1.4599579236923494,
  'max': 16.58769185933765,
  'n_over_15': 1}}

In [48]:
# ── check what was exported: read the files back and draw them ───────────────
# If the field looks wrong HERE, the remesher cannot fix it — go back to
# stage 3b rather than tuning the gradient.
V_tri, F_tri = read_obj(field["obj"])
F_tri = np.array(F_tri, dtype=np.int32)
d1 = np.loadtxt(field["d1"])
d2 = np.loadtxt(field["d2"])

# Rows refer to the triangles listed in indices.dat, or to all of them
if "indices" in field:
    tri_idx = np.loadtxt(field["indices"], dtype=int).ravel()
else:
    tri_idx = np.arange(len(F_tri))
print(f"{len(V_tri)} vertices, {len(F_tri)} triangles, "
      f"{len(d1)} frames on {len(tri_idx)} constrained triangles")
assert len(d1) == len(d2) == len(tri_idx)

tri_bary = V_tri[F_tri[tri_idx]].mean(axis=1)
s_tri = np.mean(np.linalg.norm(V_tri[F_tri[:, 1]] - V_tri[F_tri[:, 0]], axis=1)) / 3

def _seg_net(p, d, scale):
    n = len(p)
    verts = np.empty((2 * n, 3))
    verts[0::2] = p - scale * d
    verts[1::2] = p + scale * d
    edges = np.stack([np.arange(0, 2*n, 2), np.arange(1, 2*n, 2)], axis=1)
    return verts, edges

# Which triangles carry a constraint
constrained = np.zeros(len(F_tri))
constrained[tri_idx] = 1.0

#kv.register_surface_mesh("remesh_input", V_tri, F_tri, show_edges=True)
#kv.add_scalar_quantity("remesh_input", "constrained", constrained, defined_on="faces")
#kv.register_curve_network("D1", *_seg_net(tri_bary, d1, s_tri), color=(0.9, 0.2, 0.1), radius=0.0015)
#kv.register_curve_network("D2", *_seg_net(tri_bary, d2, s_tri), color=(0.1, 0.3, 0.9), radius=0.0015)
kv.show()

1200 vertices, 2242 triangles, 985 frames on 985 constrained triangles
[kayviz] Viewer already open — http://127.0.0.1:8001/?app=script


In [49]:
# ── run the remesher (headless) ──────────────────────────────────────────────
GRADIENT    = 55.0    # higher = denser quads
CONSTRAINTS = 0.1     # fraction of faces kept as hard constraints (0 < f <= 1)
EXTRACTOR   = "qex"   # 'qex' (robust at seams) or 'grid'

res = run_remesher(state, gradient=GRADIENT, constraints=CONSTRAINTS,
                   extractor=EXTRACTOR, batch=True)

if res["returncode"] == 0:
    remesh_quality(state)
    kv.register_surface_mesh("remeshed", state.V_remesh, state.F_remesh,
                             show_edges=True)
    kv.add_vector_quantity("remeshed", "l", state.l_remesh, defined_on="vertices")
    kv.show()

/home/anthony/KAUSTLocal/QuadRemesher/Quad_Remesher/build/quadRemesher /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0/remesh_input.obj --field data --d1 /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0/D1.dat --d2 /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0/D2.dat --format vec --defined_on faces --indices /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0/indices.dat --gradient 55 --constraints 0.1 --extractor qex --out /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0 --name surface_00002 --batch 

f size: 2242 x 3
  NaN in X1_def: 0
  NaN in X2_def: 0
  X1_def norm range: [1, 1]
  X2_def norm range: [1, 1]
  X1·X2 dot product range: [-6.59195e-16, 6.80012e-16]
  X1·N (should be ~0): max = 4.16334e-16
  X2·N (should be ~0): max = 3.33067e-16
  Problematic faces: 0 / 2242

=== Running MIQ Parameterization ===
  Step 1: Compu

In [ ]:
# ── too coarse / too dense? sweep the gradient and pick one ──────────────────
# Each run overwrites the OBJ, so the state ends up holding the LAST value —
# re-run the cell above at your chosen GRADIENT afterwards.
rows = sweep_gradient(state, [20, 40, 80], constraints=CONSTRAINTS,
                      extractor=EXTRACTOR)

print()
for r in rows:
    if r.get("n_faces"):
        print(f"  gradient {r['gradient']:>5} -> {r['n_faces']:>5} faces  "
              f"({100*r['quad_frac']:.0f}% quads)  mean radius {r['mean_radius']:.4f}")

In [ ]:
# ── or tune it by eye in the remesher's own viewer ───────────────────────────
# Blocks until you close the window. '+'/'-' = denser/coarser (x1.5 per press),
# 'R' = preview the quad mesh, 'S' = extract and save it, '6' = seams and
# singularities. Note the gradient you settle on and put it in GRADIENT above.
# res = run_remesher(state, gradient=GRADIENT, constraints=CONSTRAINTS,
#                    extractor=EXTRACTOR, batch=False)

In [ ]:
# ── load a previously remeshed OBJ ───────────────────────────────────────────
# load_remeshed_surface(state)                       # <save_path>/<name>_Remeshed.obj
# load_remeshed_surface(state, "/path/to/other_Remeshed.obj")
# remesh_quality(state)
# kv.register_surface_mesh("remeshed", state.V_remesh, state.F_remesh, show_edges=True)
# kv.show()

---
## 4b — Fair the remeshed mesh (Q')

The paper does not go straight from the remesher to the sphere fitting:

> *"The resulting quad mesh from above may exhibit some zigzag behaviour which
> we eliminate by letting the mesh glide over the reference surface while
> enforcing fairness of the parameter lines. In this way we obtain a fair quad
> mesh Q'."*

Only then are the `(u, v)` of `Q'` computed and the sphere congruence built.
`fair_remeshed_surface` is that step, with the weights of the original
`foot_point_bspline.py` — proximity 5 against fairness 0.01, so the mesh is
held on the surface and slides tangentially rather than being pulled off it.

It updates `state.V_remesh` in place, so stage 5 then initialises its spheres
from `Q'` instead of the raw remesh.

Notes on the terms:

- **The boundary is fixed** (`FIX_BOUNDARY_FAIR`). This is the mesh being
  deformed, so without it the patch edge drifts. It is cheap here — tangential
  motion is most of what fairing does, so pinning the boundary blocks little of
  it (measured: 10% of the parameter-line second difference removed with the
  boundary fixed against 13.6% free).
- **Edge lengths** are held near the values the remesher produced, so gliding
  cannot bunch the mesh up. On `Tunel` this is not the binding constraint
  (movement 2.52e-05 with it against 2.54e-05 without), so it costs nothing
  and guards a mesh that does want to bunch.
- **The fairness weight damps itself to zero** after `damp_iteration` steps,
  by design: fairness minimised over a whole run is mean-curvature flow and
  would shrink the mesh. It is a short smoothing burst — do not extend it.
- **`W_CORNER`** holds the angle at each boundary vertex at its initial value.
  Off by default because it is redundant while the boundary is pinned; raise it
  if you set `FIX_BOUNDARY_FAIR = False`.

In [50]:
# ── glide + fairness: raw remesh -> Q' ────────────────────────────────────────
FAIR_MAX_ITER      = 20
W_PROX_FAIR        = 5.0    # hold the mesh on the reference surface
W_GLIDE_FAIR       = 5.0    #   (closest point + tangent plane)
W_FAIR_FAIR        = 1e-2   # fairness of the parameter lines (damps to 0 by design)
W_EDGE_FAIR        = 1e-1   # keep the remesher's edge lengths
W_CORNER_FAIR      = 0.0    # boundary corner angles; redundant while pinned
FIX_BOUNDARY_FAIR  = True   # hold the patch edge exactly
# ─────────────────────────────────────────────────────────────────────────────

V_before = state.V_remesh.copy()

fair_remeshed_surface(state,
                       max_iter=FAIR_MAX_ITER,
                       w_prox=W_PROX_FAIR, w_glide=W_GLIDE_FAIR,
                       w_fair=W_FAIR_FAIR, w_edge=W_EDGE_FAIR,
                       w_corner=W_CORNER_FAIR,
                       fix_boundary=FIX_BOUNDARY_FAIR)

# l was interpolated onto the raw remesh; refresh it now the vertices moved
load_remeshed_surface(state)

kv.register_surface_mesh("remesh_raw", V_before,        state.F_remesh, show_edges=True)
kv.register_surface_mesh("remesh_fair", state.V_remesh, state.F_remesh, show_edges=True)
kv.add_scalar_quantity("remesh_fair", "moved",
                        np.linalg.norm(state.V_remesh - V_before, axis=1),
                        defined_on="vertices")
kv.show()

Mesh |V|=940, |F|=843, |E|=1782
Adding quadratic fairness for 797 vertices with 4 neighbors
Adding Laplacian fairness for 143 
Fairing (Q'): 14 iters, energy 2.7915e-03 -> 9.9184e-08
  vertices moved: median 5.560e-05  max 1.135e-03  (boundary held: 0.0e+00)
Loaded /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0/surface_00002_Remeshed.obj
  940 vertices, 843 faces  843x4-gon
[kayviz] Viewer already open — http://127.0.0.1:8001/?app=script


---
## 5 — Post-optimisation

In [57]:
# ── parameters (paper Sec. 5.2 / Table 3) ─────────────────────────────────────
POSTOPT_MAX_ITER = 30
WARMUP_ITERS     = 10     # spheres-only warm-up before the vertices are freed

W_SPHERE   = 1.0          # E_sphere   : spheres pass through their face vertices
W_UNIT     = 10.0         # E_unit     : <B,B> - 4AC = 1. WITHOUT THIS (A,B,C)=0
                          #              is a global minimum of E_sphere.
W_SUPPORT  = 1e-1         # E_supp     : sphere-centre rings planar
W_TPLANE   = 1e-1         # E_torsal plane : node axes coplanar with their edge
W_PROX     = 1e-4         # E_prox_f   : vertices -> reference surface (point)
W_GLIDE    = 1e-4          # E_prox_f   : vertices -> reference surface (tangent)
W_PROX_C   = 1e-5         # E_prox_C   : sphere centres -> optimised congruence
W_FAIR     = 1e-1         # E_fair     : Chakana's quad-mesh fairness
W_REG_V    = 0.0          # E_reg      : step dampening, OFF on purpose --
W_REG_N    = 0.0          #              see below
FIX_BOUNDARY = True      # hold the remesh boundary vertices exactly fixed
# ─────────────────────────────────────────────────────────────────────────────
# E_reg is off because with adaptive_mu=False every LM step is accepted, so
# StepControl's residual (X - prev) is identically zero: it cannot move the
# optimum, it only shrinks the step, which LM's own mu already does. Measured
# on Tunel at 80 iterations, |dist-r| was 1.65e-05 with it off against 2.29e-04
# at the paper's 0.04/0.05, and 300 damped iterations still only reached
# 1.01e-04. Raise them if adaptive damping is ever switched on.
#
# FIX_BOUNDARY is a hard constraint (Optimizer.fix_variables), not a penalty:
# those vertices get exactly zero step. It is off by default because it is a
# trade-off, not a free win — on Tunel over 80 iterations it held the boundary
# bit-identical but cost 5.7x on the sphere fit (|dist - r| 1.56e-05 ->
# 8.93e-05), since boundary vertices do need to move for their own faces'
# spheres to be fitted. Most useful for a closing few iterations, or when the
# boundary has to match a neighbouring patch.

setup_postopt_optimizer(state,
                         w_sphere=W_SPHERE, w_unit=W_UNIT,
                         w_support=W_SUPPORT, w_tplane=W_TPLANE,
                         w_prox=W_PROX, w_glide=W_GLIDE, w_prox_c=W_PROX_C,
                         w_fair=W_FAIR, w_reg_v=W_REG_V, w_reg_n=W_REG_N,
                         warmup_iters=WARMUP_ITERS, fix_boundary=FIX_BOUNDARY)
run_postopt_optimizer(state, max_iter=POSTOPT_MAX_ITER)

# How well did it do?  |dist - r| is the geometric form of E_sphere: how far
# each sphere is from passing through its own four face vertices.
from moebius.utils.spheres import center_radius_from_implicit
A, B, C = state.postopt.unpack("A", "B", "C")
v = state.postopt.unpack("v").reshape(-1, 3)
c, r, is_plane = center_radius_from_implicit(A, B.reshape(-1, 3), C)
err = np.array([np.abs(np.linalg.norm(v[f] - c[i], axis=1) - r[i]).max()
                for i, f in enumerate(state.F_remesh_quads)])
print(f"\n|dist - r|: median {np.median(err):.3e}  max {err.max():.3e}")
print(f"radii: median {np.median(r):.4f}   planes (|A|~0): {int(is_plane.sum())}")

Mesh |V|=940, |F|=843, |E|=1782
Warm-up: 10 sphere-only steps, energy 9.7416e-03 -> 1.1275e-03
Adding quadratic fairness for 797 vertices with 4 neighbors
Adding Laplacian fairness for 143 
Holding 192 boundary vertices fixed (20% of the mesh)
Thread pool: 12 workers (terms=12, cpus=32)
Starting fixed-damping LM optimization (max_iter=30, tol=1e-08)
Levenberg-Marquardt Optimization (Simple)
Iter   | Energy        | Norm Energy   | ||dx||        | μ             | ρ        | Time    
------------------------------------------------------------------------------------------
1      | 2.631881e-01  | 4.990142e-05  | 5.524103e+00  | 6.626497e-04  | N/A      | 1.2936  
10     | 9.466985e-03  | 1.821101e-07  | 2.061973e-01  | 6.626497e-04  | N/A      | 1.2487  
20     | 3.136173e-04  | 9.602136e-08  | 9.678563e-02  | 6.626497e-04  | N/A      | 0.8142  
30     | 2.216821e-04  | 6.839653e-08  | 6.590458e-02  | 6.626497e-04  | N/A      | 0.7182  
Reached maximum iterations (30)

 WEIGHTS REPORT
T

In [58]:
# ── visualise post-opt result ─────────────────────────────────────────────────
V_opt = state.postopt.unpack("v").reshape(-1, 3)
A, B, C = state.postopt.unpack("A", "B", "C")
B_r = B.reshape(-1, 3)

# Panels straight from the implicit coefficients. Passing (A, B, C) rather than
# centres and radii is what lets a face whose sphere degenerates to a plane be
# drawn FLAT rather than as a huge sphere: once converted, c = B/(2A) and
# r = 1/(2A) have both run off and come back as large finite numbers. Planes
# are rare here — stage 1b pushes |H| above a threshold and r = 1/H, so flat
# regions are excluded from the start — but the panel code handles them anyway.
pV, pF = panels_from_spheres(V_opt, state.F_remesh_quads, A, B_r, C,
                              target_triangles=120)

centers, radii, is_plane = center_radius_from_implicit(A, B_r, C)
print(f"{len(radii)} faces: {int(is_plane.sum())} plane(s), "
      f"|r| median {np.median(np.abs(radii[~is_plane])):.4f}")

#kv.register_surface_mesh("optimised_mesh", V_opt, state.F_remesh_quads)
kv.register_surface_mesh("sphere_panels",  pV,    pF)
#kv.add_scalar_quantity("optimised_mesh", "sphere_radius", np.abs(radii),                       defined_on="faces")
kv.show()

843 faces: 0 plane(s), |r| median 0.3511
[kayviz] Viewer already open — http://127.0.0.1:8001/?app=script


---
## 6 — Save results

In [59]:
save_results(state)
print("Saved to:", state.save_path)

Results saved to /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0
Saved to: /home/anthony/KAUSTLocal/Moebius_project/notebooks/out/surface_00002/experiment0


---
## Utilities — reload & inspect

```python
# Reload a saved state
state = load_state("../notebooks/out/CMC/experiment0/state.pickle")

# Plot energy history
plt.figure(figsize=(8, 4))
plt.semilogy(state.lc_opt.energy,     label="LC")
plt.semilogy(state.torsal_opt.energy, label="Torsal")
plt.legend(); plt.xlabel("Iteration"); plt.ylabel("Energy"); plt.grid()
```